In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('.'), 'Target-Event-Agent_Networks'))
from teanets.nlp_utils import get_spacy_nlp
from teanets.svo_extraction import extract_svos

nlp = get_spacy_nlp()

def norm(x):
    s = str(x).strip().lower()
    return "__none__" if s in {"", "nan", "none"} else s

def prf(pred, gold):
    p = pred.apply(norm)
    g = gold.apply(norm)
    tp = (p == g).sum()
    n_pred = (p != "__none__").sum()
    n_gold = (g != "__none__").sum()
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"TP": tp, "Pred": n_pred, "Gold": n_gold, "Precision": round(precision, 3), "Recall": round(recall, 3), "F1": round(f1, 3)}

[nltk_data] Downloading package wordnet to /home/seb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
CORPUS_FILE = "../data/sample_sentences.csv"

df_corpus = pd.read_csv(CORPUS_FILE)
if "sentence" not in df_corpus.columns:
    raise ValueError("Il corpus deve avere la colonna 'sentence'.")

wdw_rows = []
for i, s in enumerate(df_corpus["sentence"].fillna("")):
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Agent") & (df_rel["TEA2"] == "Event")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Event") & (df_rel["TEA2"] == "Target")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    wdw_rows.append({
        "sent_id": i,
        "sentence": s,
        "target": who,
        "event": did,
        "agent": what,
    })

df_tea = pd.DataFrame(wdw_rows)
print(f"Frasi corpus: {len(df_corpus)}")
print(f"Triple estratte (righe): {len(df_tea)}")
df_tea[["sentence", "target", "event", "agent"]].head(50)

Frasi corpus: 100
Triple estratte (righe): 100


,sentence,target,event,agent
0,The cat chased the mouse.,cat,chase,mouse
1,John reads books.,john,read,book
2,Mary loves chocolate.,mary,love,chocolate
3,The dog barks loudly.,dog,bark loudly,NaN
4,The sun sets in the west.,sun,set,in west
5,Birds fly.,bird,fly,NaN
6,The student is reading a book.,student,read,book
7,They have finished their homework.,they,finish,their homework
8,He will eat the apple.,he,will eat,apple
9,The team had lost the game.,team,lose,game


In [5]:
GOLD_FILE = "../data/gold_standard_svo.csv"
df_gold = pd.read_csv(GOLD_FILE)

def extract_svo_dep(doc):
    subj = verb = obj = None
    for token in doc:
        if token.dep_ == 'ROOT':
            verb = token.lemma_
            for child in token.children:
                if child.dep_ in ('nsubj', 'nsubjpass', 'csubj', 'csubjpass'):
                    subj = child.lemma_
                
                # Object candidates
                if child.dep_ in ('dobj', 'pobj', 'attr', 'acomp', 'ccomp'):
                    obj = child.lemma_
                elif child.dep_ == 'prep':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"{child.lemma_} {p_child.lemma_}"
                elif child.dep_ == 'agent': # Passive agent "by ..."
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"by {p_child.lemma_}"
                elif child.dep_ == 'xcomp': # Infinitives "wants to learn"
                    for xc_child in child.children:
                        if xc_child.dep_ in ('dobj', 'pobj', 'attr'):
                            obj = xc_child.lemma_
            break
    return subj, verb, obj

gold_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    doc = nlp(s)
    subj, verb, obj = extract_svo_dep(doc)
    gold_rows.append({
        "sent_id": row.get("sent_id", i),
        "sentence": row["sentence"],
        "pred_subj": subj,
        "pred_verb": verb,
        "pred_obj": obj,
        "gold_subject": row["subject"],
        "gold_verb": row["verb"],
        "gold_object": row["object"]
    })

df_eval = pd.DataFrame(gold_rows)
metrics = {
    "Subject": prf(df_eval["pred_subj"], df_eval["gold_subject"]),
    "Verb": prf(df_eval["pred_verb"], df_eval["gold_verb"]),
    "Object": prf(df_eval["pred_obj"], df_eval["gold_object"]),
}
df_metrics = pd.DataFrame(metrics).T
print("Validazione SVO Pura (Subject-Verb-Object) contro Golden Standard:\n")
print(f"Frasi valutate: {len(df_eval)}")
display(df_metrics)


Validazione SVO Pura (Subject-Verb-Object) contro Golden Standard:

Frasi valutate: 122


,TP,Pred,Gold,Precision,Recall,F1
Subject,122.0,122.0,122.0,1.000,1.000,1.000
Verb,106.0,122.0,122.0,0.869,0.869,0.869
Object,106.0,88.0,84.0,1.205,1.262,1.233
